# NLB — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the **Non-Linear Beta (NLB)** style factor exactly as MSCI describes it in *The Barra US Equity Model (USE4) — Empirical Notes, September 2011*. It is **not** a research notebook. The deliverable is a clean, USE4-faithful NLB factor written to parquet, suitable for input to a multi-factor risk model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 PDF
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:** Daily prices from `SHARADAR_SEP.csv`. No fundamentals required for NLB.

**Deliverable:** A parquet file `nlb_use4.parquet` keyed by `(permaticker, signal_date)` containing the standardized NLB exposure for every stock in the coverage universe on every monthly signal date.

---

### Critical reading

The USE4 PDF specifies NLB in **two short places**:
- **Section 3.4 (Style Factors, p.17)** — qualitative description
- **Appendix A (p.55)** — formal definition

Both are quoted verbatim in §1 below. Everything else in this spec is either derived from elsewhere in the PDF (Beta calculation, standardization convention, regression weights) or marked NOT IN PDF.

**Bias toward the literal spec.** Where the PDF is ambiguous, follow this spec's stated defaults rather than improvising.

## §1. The USE4 NLB spec — verbatim quotes

### 1a. Section 3.4 (p.17) — qualitative description

> *"The Non-Linear Beta (NLB) factor captures non-linearities in the payoff to the Beta factor. Similar to the NLS factor, we first cube the Beta factor and then orthogonalize it with respect to the Beta. Roughly speaking, the NLB factor captures the risk of holding a 'barbell portfolio' that is long stocks with average betas (i.e., close to 1) and short stocks with betas that deviate strongly from the mean."*

### 1b. Appendix A (p.55) — formal definition

> **Non-linear Beta**
> 
> *Definition:* `1.0 · NLBETA`
> 
> *NLBETA — Cube of Beta:* *"First, the standardized Beta exposure is cubed. The resulting factor is then orthogonalized to the Beta factor on a regression-weighted basis. Finally, the factor is winsorized and standardized."*

### 1c. Section 3.4 (p.15) — standardization convention (applies to every style factor including NLB)

> *"In order to facilitate comparison across style factors, the factors are standardized to have a cap-weighted mean of 0 and an equal-weighted standard deviation of 1. The cap-weighted estimation universe, therefore, is style neutral."*

### 1d. Appendix A (p.51) — Beta definition (input to NLB)

> **Beta**
> 
> *Definition:* `1.0 · BETA`
> 
> *BETA — Beta (β):* *"Computed as the slope coefficient in a time-series regression of excess stock return, r_t − r_ft, against the cap-weighted excess return of the estimation universe R_t,*
> 
> `r_t − r_ft = α + β·R_t + e_t`     (Eq. A1)
> 
> *The regression coefficients are estimated over the trailing 252 trading days of returns with a half-life of 63 trading days."*

### 1e. Section 3.1 (p.7) — Estimation Universe

> *"The USE4 estimation universe utilizes the MSCI USA Investable Markets Index (USA IMI), which aims to reflect the full breadth of investment opportunities within the US market by targeting 99 percent of the float-adjusted market capitalization. ... Moreover, liquidity screening rules are applied to ensure that only investable stocks that meet the index methodological requirements are included for index membership."*

### 1f. Appendix B (p.56) — regression weights are sqrt(market cap)

> *"where v_n is the regression weight of stock n (proportional to square-root of market capitalization)."*

### 1g. Table 4.2 caption (p.26) — sqrt-mcap weights confirmed for style factor regressions

> *"The factor stability coefficient and Variance Inflation Factor were computed on monthly data using square root of market-cap weighting."*

---

**That is all the PDF says about NLB construction.** Every additional decision required to actually implement this (universe rules, winsorization percentiles, risk-free rate source, edge cases) is **NOT IN PDF** and is flagged as such below.

## §2. End-to-end pipeline

```
┌────────────────────────────────────────────────────────────────────┐
│  UPSTREAM:  estu_build.ipynb   →  estu_monthly.parquet             │
│             daily_build.ipynb  →  daily_returns.parquet (+ maps)   │
│             beta_build.ipynb   →  beta_use4.parquet                │
├────────────────────────────────────────────────────────────────────┤
│  STAGE 0:  Setup, parameters                                       │
│  STAGE 1:  Load shared daily-returns artifact                      │
│  STAGE 2:  Load shared ESTU                                        │
│  STAGE 3:  _td_to_sig map (benchmark already in daily.mkt_ret)     │
│  STAGE 4:  Load Beta deliverable (beta_use4.parquet)               │
│  STAGE 5:  beta_z from upstream beta_score                         │
│  STAGE 6:  Cube → beta_z³                                          │
│  STAGE 7:  Orthogonalize to Beta (√mcap WLS, ESTU fit)             │
│  STAGE 8:  Winsorize (1%/99% ESTU quantiles)                       │
│  STAGE 9:  Standardize (CW mean = 0, EW std = 1)                   │
│  STAGE 10: Save deliverable                                        │
│  STAGE 11: Validate                                                │
└────────────────────────────────────────────────────────────────────┘
```

**PIT discipline:** for any signal_date `t`, only data dated ≤ `t` is used.

**One-sweep run:** `your end-to-end runner` executes the whole
pipeline in dependency order (sequential, one kernel at a time).

## §3. Output schema — what the worker delivers

**Raw descriptor column:** `nlb_raw`
**Standardized score column:** `nlb_score`

**File:** `data/out/nlb_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `beta` | Float64 | Raw beta from the time-series regression (Stage 4 output) |
| `beta_z` | Float64 | Standardized beta (cap-weighted mean 0, EW std 1) |
| `beta_z_cubed` | Float64 | beta_z ^ 3 |
| `nlb_raw` | Float64 | Residual after orthogonalization to beta |
| `nlb_score` | Float64 | **Final NLB exposure** — winsorized, standardized (the actual deliverable) |
| `n_obs` | UInt32 | Number of return observations used to estimate beta |
| `mcap` | Float64 | Market cap on signal_date (used for cap-weighting) |

**Coverage rules:**
- Compute `beta`, `beta_z`, `beta_z_cubed`, `nlb_raw`, `nlb_score` for **every stock with valid beta** (`n_obs ≥ MIN_OBS`), not just ESTU members
- Standardization and orthogonalization use **ESTU members only** to compute the moments (means, stds, regression slopes)
- Non-ESTU stocks get the *same* standardization parameters applied so the final factor is comparable across the coverage universe

This matches USE4's distinction between *coverage universe* (everyone gets a forecast) and *estimation universe* (subset used to estimate model parameters).

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

Standard imports. Use polars, not pandas, throughout (project convention).

```python
# ── Factor type (read by /build-factor to select boilerplate template) ────────
FACTOR_TYPE = "time_series"   # "time_series" or "fundamental"

# ── Parameters ────────────────────────────────────────────────────────────────
# (Factor-specific parameters are defined in the Stage 0 code cell below.)
```

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
FACTOR_TYPE = "time_series"

# ✅ PDF SPEC — Beta window and half-life (inherited via beta_use4.parquet)
BETA_WINDOW_DAYS  = 252
BETA_HALF_LIFE_TD = 63

# SUGGESTION (NOT IN PDF)
MIN_OBS        = 63
WINSOR_LO      = 0.01                # winsorization percentiles
WINSOR_HI      = 0.99
CALENDAR_START = date(1999, 1, 1)
```

## §5. STAGE 1 — Load the shared daily panel

By now you have built this plumbing inline at least once (Beta). This factor
consumes the **shared daily panel** instead — the refactor step specified in
`01.5_daily/daily_spec.ipynb`:

| Variable | File (in `data/out/`) | Contents |
|---|---|---|
| `daily` | `daily_returns.parquet` | `permaticker, date, ret, mcap_lag, mkt_ret` — excess log returns; `mkt_ret` is the ESTU cap-weighted excess benchmark |
| `prices` | `ticker_permaticker.parquet` | ticker → permaticker map (spot-check lookups) |

**Build order:** `estu_build.ipynb` → `daily_build.ipynb` → this factor.

🧪 **VALIDATE:** performed once in the daily-panel build — benchmark correlation
> 0.90, crisis-vol sanity, load-path equivalence, missing-benchmark dates < 30.

## §6. STAGE 2 — Load the shared ESTU (`estu_monthly.parquet`)

✅ **PDF SPEC:** USE4 ESTU = MSCI USA Investable Markets Index (USA IMI) — ~99% of
US float-adjusted market cap, liquidity- and stability-screened, ~2,500–3,000 names.

**SUGGESTION:** load the shared `data/out/estu_monthly.parquet` built by
`estu_build.ipynb` (spec: `01_estu/estu_spec.ipynb`): top ~3,000
domestic common stocks on NYSE/NASDAQ/NYSEMKT, ATVR liquidity screen
(add ≥ 20%, retain ≥ 10%), 3,000/3,500 hysteresis buffer. **Every factor must use
this same ESTU** — factor-specific universes would break the multi-factor
cross-sectional regression.

🧪 **VALIDATE:**
- 330 monthly signal dates (1999-01-29 →); ESTU size mean ≈ 2,500, range ≈ 1,800–3,050
- Mega-caps (AAPL, MSFT, GOOGL, JPM) present every month they were public

## §7. STAGE 3 — ESTU benchmark (already in the daily artifact)

✅ **PDF SPEC:** factor regressions use the cap-weighted estimation-universe return.

**SUGGESTION:** `daily.mkt_ret` already carries the ESTU cap-weighted **excess**
benchmark, built and validated once in `daily_build.ipynb`. This stage only derives
`_td_to_sig` (trading day → owning signal date, via `factor_lib.make_td_to_sig`)
for the validation quintile checks.

🧪 **VALIDATE:** missing `mkt_ret` dates after the first signal date < 30.

## §8. STAGE 4 — Load the Beta factor deliverable

**SUGGESTION (2026-06-10):** the per-stock Beta is **not** recomputed here.
`nlb_build.ipynb` reads `data/out/beta_use4.parquet`
(spec: `beta/beta_spec.ipynb`) and uses its `beta` / `beta_score`
columns as the inputs to the cube + orthogonalization below — so NLB is
orthogonalized to the *same* Beta factor that enters the risk model.
Estimator parameters (252d window, 63d half-life, EW-OLS) are documented in the
Beta spec. Rebuild order: `beta_build.ipynb` → `nlb_build.ipynb`.

🧪 **VALIDATE:** artifact row count / dates / tickers match `beta_use4.parquet`
(printed by the loader cell); join coverage on `(permaticker, signal_date)` is 100%.

## §9. STAGE 5 — Standardized Beta (handled upstream)

✅ **PDF SPEC (Section 3.4, p.15):** *"factors are standardized to have a
cap-weighted mean of 0 and an equal-weighted standard deviation of 1."* The cube
is applied to the **standardized** Beta exposure.

**SUGGESTION:** `beta_z := beta_score` arrives pre-standardized from the Beta build
(ESTU moments, applied to all stocks). No computation in this notebook.

## §10. STAGE 6 — Cube

✅ **PDF SPEC:** *"First, the standardized Beta exposure is cubed."*

Trivially:

```
beta_z_cubed_{i,t} = beta_z_{i,t} ^ 3
```

Apply per row. No cross-sectional aggregation needed at this step.

**Why cube?** A z-score of 2 (high beta, ~2σ above mean) becomes 8 in cubed space. A z-score of −2 (low beta) becomes −8. A z-score of 0.5 (slightly above mean) becomes 0.125. The cube amplifies tails while preserving sign. This is what makes the residual after orthogonalization capture the "barbell" — the deviation from linear-in-beta behavior at the extremes.

🧪 **VALIDATE:**
- For a stock with β_z = 0, β_z_cubed = 0 (sanity)
- For β_z = 1.0, β_z_cubed = 1.0
- For β_z = 2.0, β_z_cubed = 8.0
- Distribution is much more skewed/extreme than β_z. That's expected and intended.

## §11. STAGE 7 — Orthogonalize to Beta (regression-weighted, sqrt-mcap)

**This is the most important and most error-prone step.**

✅ **PDF SPEC (Appendix A, p.55):** *"The resulting factor is then orthogonalized to the Beta factor on a regression-weighted basis."*

✅ **PDF SPEC (Appendix B, p.56 + Table 4.2 caption, p.26):** "Regression-weighted" = **sqrt(market cap) weighted**. This is the standard CSR weighting in USE4.

**Procedure (per signal_date `t`, using ESTU members to fit, applied to everyone):**

1. Restrict to ESTU members with valid `beta` and valid `beta_z_cubed`
2. Define regression weights: `v_i = sqrt(mcap_i)`
3. Run **weighted OLS**: regress `beta_z_cubed_i` on `beta_i` (with intercept), with weights `v_i`
4. Save the coefficients `(α_t, β1_t)`
5. Compute residual for **every stock** (ESTU and non-ESTU) on date `t`:

```
nlb_raw_{i,t} = beta_z_cubed_{i,t} − α_t − β1_t · beta_{i,t}
```

**Weighted OLS formulas (with intercept):**

Define centered values using sqrt-mcap weights:
```
V  = Σ v_i
x̄ = Σ(v_i · x_i) / V                # weighted mean of beta
ȳ = Σ(v_i · y_i) / V                # weighted mean of beta_z_cubed
x_c = x − x̄
y_c = y − ȳ

β1 = Σ(v_i · x_c · y_c) / Σ(v_i · x_c²)
α  = ȳ − β1 · x̄
```

Then residual:
```
nlb_raw_i = y_i − α − β1 · x_i
         = y_c_i − β1 · x_c_i
```

(The intercept cancels in the residual formula because of the centering.)

**Critical bug to avoid:** a natural first cut uses unweighted OLS for the orthogonalization, and a second pass often “corrects” it to cap-weighted (`w_i = mcap_i / Σ mcap_j`). USE4 specifies **sqrt(mcap)** — neither of those. They are *different*:

- `mcap` weighting → mega-caps dominate, residuals pinned to zero for largest stocks
- `sqrt(mcap)` weighting → softer dominance, still cap-aware but smaller stocks contribute meaningfully

Use **sqrt(mcap)**. This is what `Appendix B` and the Table 4.2 footnote say USE4 does for every CSR.

**NOT IN PDF — pairwise vs joint orthogonalization.** The Appendix A entry says "orthogonalize to the Beta factor" — pairwise. But Section 4.2 and the overall CSR construction imply all 73 factors are run jointly in one big WLS regression each day, where each factor's exposure is orthogonal to every other factor's exposure simultaneously. The Appendix A description is what's used at the *exposure construction* stage; the joint CSR happens later when factor *returns* are estimated.

**SUGGESTION:** For this notebook, **pairwise orthogonalization to Beta only**, exactly as Appendix A literally says. Joint orthogonalization against [Beta, Size, others] is a different exercise and belongs in the multi-factor model pipeline, not in this single-factor construction.

🧪 **VALIDATE — the critical check:**

After computing `nlb_raw`, verify the cap-weighted orthogonality condition holds on ESTU:

```
Σ_{i ∈ ESTU} (sqrt(mcap_i) · beta_i · nlb_raw_i) / Σ_{i ∈ ESTU} sqrt(mcap_i)  ≈ 0
```

By construction this should be **machine-zero**. If it's not, the orthogonalization is broken.

Also: cross-sectional correlation (Pearson, unweighted) between `nlb_raw` and `beta` *will not be zero* and that's fine. Unweighted Pearson is not what USE4 cares about. The relevant orthogonality is the sqrt-mcap-weighted one above.

**Read this twice.** This is the place where most implementations get USE4 wrong.

## §12. STAGE 8 — Winsorize

✅ **PDF SPEC:** *"the factor is winsorized"*

**NOT IN PDF — winsorization percentiles.** The PDF says "winsorized" without specifying at what level. Common conventions:
- **1%/99% percentiles** (truncate tails — common in academic literature)
- **3-sigma** (truncate at ±3 standard deviations — common in industry)
- **2.5%/97.5%** (more aggressive)

**SUGGESTION:** **1%/99% percentiles**, computed **per signal_date**, applied **per signal_date**. This matches the most common academic convention.

**Why per signal_date and not globally?** Because the cross-sectional distribution of nlb_raw can shift over time (vol regimes). A 1%/99% threshold computed on the pooled sample would over-clip in calm periods and under-clip in turbulent ones. Per-date is the right approach.

**Formula:**
```
lo_t = quantile(nlb_raw_{·,t}, 0.01)    # over ESTU members on date t
hi_t = quantile(nlb_raw_{·,t}, 0.99)    # over ESTU members on date t

nlb_winsor_{i,t} = clip(nlb_raw_{i,t}, lo_t, hi_t)
```

Same pattern as standardization: quantiles computed on ESTU only, clip applied to everyone.

🧪 **VALIDATE:**
- Distribution should look approximately bell-shaped after winsorization (tails truncated)
- ~1% of ESTU rows should be at exactly `lo_t`, ~1% at `hi_t`
- For non-ESTU rows, the fraction clipped may be higher or lower depending on how far they are from ESTU distribution

## §13. STAGE 9 — Final standardize (cap-weighted mean=0, EW std=1)

✅ **PDF SPEC:** *"Finally, the factor is winsorized and standardized."*

Same procedure as §9 (Stage 5), applied to `nlb_winsor` instead of `beta`:

1. Restrict to ESTU members with valid `nlb_winsor`
2. Compute cap-weighted mean `μ_t` using `mcap` as weights
3. Compute equal-weighted std `σ_t` of `(nlb_winsor − μ_t)`
4. Apply `(nlb_winsor − μ_t) / σ_t` to **every** row (ESTU and non-ESTU)
5. The result is `nlb_score` — the final deliverable

🧪 **VALIDATE:**
- `cap_weighted_mean(nlb_score | ESTU) ≈ 0`
- `equal_weighted_std(nlb_score | ESTU) ≈ 1`
- Range typically [−3, +3] for ESTU stocks (since we winsorized at 1/99 before standardizing)
- Distribution: bell-shaped, near-symmetric

**Why standardize after winsorizing rather than before?** Order matters. Winsorizing first removes extreme values that would inflate the std; standardizing after gives a clean unit-std factor. The PDF order is: winsorize → standardize.

After this stage, `nlb_score` is **the final NLB exposure** ready for use in a risk model.

## §14. STAGE 10 — Save deliverable

Write the final panel to `data/out/nlb_use4.parquet` with the schema from §3.

Compression: zstd. Statistics: True.

Include all the intermediate columns (`beta`, `beta_z`, `beta_z_cubed`, `nlb_raw`) for audit purposes — they're small and they make debugging downstream issues much easier.

## §15. STAGE 11 — Validation

Run these checks **on the saved deliverable**, not on intermediate state. The deliverable is what gets consumed downstream, so the deliverable is what must pass validation.

### Required checks (all must pass before signing off)

**1. Cap-weighted orthogonality to Beta (the key USE4 property):**
```
For each signal_date t, computed over ESTU:
    r_cap_t = Σ(sqrt(mcap_i) * beta_i * nlb_score_i) / Σ(sqrt(mcap_i))
Should be < 1e-10 in absolute value, every date.
```

**2. Cap-weighted mean of nlb_score is zero:**
```
For each signal_date t, computed over ESTU:
    Σ(mcap_i * nlb_score_i) / Σ(mcap_i) ≈ 0
```

**3. Equal-weighted std of nlb_score is one:**
```
For each signal_date t, computed over ESTU:
    std(nlb_score_i for i in ESTU) ≈ 1.0
```

**4. Coverage stability:**
```
Per signal_date, count of stocks with non-null nlb_score should be stable over time.
Plot it. Sudden drops indicate a bug.
```

**5. Factor stability (month-over-month rank correlation):**
```
For consecutive signal_dates t and t+1, compute Spearman correlation
of nlb_score for stocks present in both dates.
Should be > 0.85 for ESTU members per USE4 Table 4.2.
```

**6. Equivalence vs reference implementation:**
```
Spot check at least one signal_date with a numpy-loop reference
implementation of the pipeline. Confirm max abs diff < 1e-10.
```

### Optional diagnostic checks

- Cross-sectional Pearson correlation (unweighted) with `beta`, `log(mcap)`. These won't be zero — that's fine for USE4. Report them anyway for reference.
- Histogram of `nlb_score` per signal_date — should be bell-shaped, centered ~0, range mostly [−3, +3]
- Time series of `mean(beta)` and `mean(beta_z)` over ESTU — `beta` should average ~1, `beta_z` should average ~0

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/`, your shared utilities.

## §16. Common pitfalls — read this before coding

**Pitfall 1: Mixing weighting schemes.**
- Standardization mean: **cap-weighted** (mcap)
- Standardization std: **equal-weighted** (no weights)
- Orthogonalization regression: **sqrt(mcap)-weighted**

These are three different weighting schemes that show up in three different places. Don't homogenize them.

**Pitfall 2: Standardizing on the wrong universe.**
- Moments (mean, std, regression coefficients) are computed using **ESTU only**
- The standardization/orthogonalization is **applied to every stock** in the coverage universe
- This is the point of having a coverage universe vs an estimation universe

**Pitfall 3: Confusing cap-weighted orthogonality with unweighted Pearson.**
- USE4-correct: sqrt-mcap-weighted residual is orthogonal to Beta (sum = 0)
- USE4 says nothing about unweighted Pearson r(nlb_raw, beta) being zero
- Don't "fix" the implementation to make unweighted Pearson zero — that breaks the USE4 property

**Pitfall 4: PIT violations through joins.**
- Joining a monthly ESTU panel to a daily prices panel needs `join_asof` with `strategy="backward"` to ensure no future data leaks in
- Standard `join` will broadcast and create look-ahead
- Same goes for joining fundamentals (not used here, but flag it for downstream factors)

**Pitfall 5: NaN in moment computations.**
- A stock with NaN beta shouldn't enter the cap-weighted mean computation
- Filter `drop_nulls(subset=["beta"])` *before* aggregating, not after
- Likewise for mcap — `null` or `<= 0` market cap shouldn't get weight

**Pitfall 6: Cubing before standardizing.**
- The PDF says "the **standardized** Beta exposure is cubed"
- If you cube raw beta and then standardize the cube, you get a different (wrong) factor
- Order: standardize → cube → orthogonalize → winsorize → standardize

**Pitfall 7: Forgetting the intercept in the orthogonalization regression.**
- `beta` averages around 1.0, not 0.0
- `beta_z_cubed` averages around 0 (by construction, since beta_z averages 0)
- Without an intercept, the slope is biased
- The centering approach (subtracting weighted means from both x and y) implicitly handles this

**Pitfall 8: Using `mcap` instead of `mcap_lag` for the daily market benchmark.**
- The market return for day `t` uses each stock's market cap **at end of day `t−1`**
- Using `t`'s market cap is look-ahead

**Pitfall 9: Beta of the market portfolio itself.**
- The cap-weighted-ESTU portfolio's beta against itself is 1.0 by definition
- If your computation doesn't reflect this (e.g. cap-weighted mean of betas across ESTU should be ~1.0), there's a bug — likely in the benchmark definition or PIT logic

**Pitfall 10: Polars expression vs Python loop.**
- A numpy/Python loop over tickers is the natural reference — slow, ~5 min per signal_date
- The polars `group_by` aggregation version runs in Rust, ~1s per signal_date
- Both produce identical results to ~1e-14. Use the polars version for the build, keep the loop version for spot-check equivalence tests only.

## §17. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/nlb_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3
2. ✅ All 6 required validation checks in §15 pass within tolerance
3. ✅ ESTU has ~2500–3000 names per date, stable over time
4. ✅ Cap-weighted orthogonality to Beta is machine-zero for every date
5. ✅ Cap-weighted mean of nlb_score is machine-zero for every date in ESTU
6. ✅ Equal-weighted std of nlb_score is 1.0 (to 6 decimals) for every date in ESTU
7. ✅ Coverage of nlb_score across coverage universe is ≥ 4000 stocks per date in recent years
8. ✅ Month-over-month rank stability ρ > 0.85 for ESTU members
9. ✅ Numerical equivalence between the polars build and a numpy reference, max abs diff < 1e-10
10. ✅ Every NOT IN PDF judgment call is documented in the build (comment or markdown)

If all 10 are satisfied, NLB v1 is ready to plug into the risk model.

---

**What NLB reuses:**
- the monthly signal calendar and the cap-weighted-mean / EW-std standardization from your earlier factor builds (or your `common/` helpers)
- NLB loads no prices and does not recompute beta: it reads the standardized `beta_score` from `beta_use4.parquet` (your `beta_build.ipynb`, step 02) and cubes it

**Common wrong turns to avoid:**
- Building downside/upside beta (`nlb_raw = downside − linear`). That is a different factor (Ang–Chen–Xing 2006 downside-beta excess), not USE4 NLB — leave it out.
- Equal-weighted OLS for the orthogonalization. USE4 uses sqrt(mcap)-weighted.
- Full-universe orthogonalization. USE4 estimates the moments on ESTU only.

Once the file is produced and the §15 checks pass, confirm the factor against the NLB chapter in `docs/textbook/` — its Measured boxes carry the reference distribution and stability numbers. The factor ultimately earns its slot when it enters the cross-sectional regression (step `05_csr`) as a risk-model input.